# ClashAI FV3 generalist with Training Camp card coverage

This fresh Colab run trains a broad generalist from 93 replay reconstructions: 68 Archers-containing replays with at least some simulator-accepted plays, plus the 25-replay IceBow corpus. The combined FV3 dataset has 22,007 rows, 6,565 play rows, 1,634 held-out rows, 95 decks, and 114 card identities. All eight cards in the recorded GPGPC Training Camp deck are represented in the vocabulary and have play examples.

The source does not contain the exact eight-card GPGPC deck. Of 5,484 source card plays in the 68 Archers matches, 4,431 were simulated; 40 of those 68 replays end before all source plays were simulated. Across those matches 293 ability events were omitted. Replay events were reconstructed through RoyaleSim, so these are not original screenshot states. The model is intended for offline inference and held-out evaluation; it is not approved for live play. Held-out support for rare deck cards is limited and is reported in the run manifest.

Before running: upload `clashai_pipeline_bundle.zip` and `dataset-training-camp-generalist-fv3.npz` to `MyDrive/OpenCRBOT/`. Select a T4 GPU or another available GPU. This trains from scratch and does not load the earlier IceBow checkpoint.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib, json, os, subprocess, sys, zipfile

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/OpenCRBOT')
DATA = DRIVE / 'dataset-training-camp-generalist-fv3.npz'
CODE_ZIP = DRIVE / 'clashai_pipeline_bundle.zip'
EXPECTED_DATA_SHA256 = 'c4c59f52ede3ff7043cc46147ddf6fdab2104485e565d0ab2417c5bfdf4b27fd'
EXPECTED_CODE_SHA256 = '0c768918cca15be966bbf6b902be113a8bccaaba8be6c93278e5d99c83dc9a5e'
SOURCE_CORPUS_MANIFEST_SHA256 = {
    'archers_replays': 'a320e777abb76af9618a7fd56aa27fb1b7cef802bc68e8d3054301b6b0253eba',
    'icebow_replays': '818ba57895228e30674d41c1c13c45edb285ff6c203d642669dab7cd2a55af98',
}
SOURCE_SELECTION_SHA256 = '8e356c7a2f7a9549e8c9ca3b61bcfd22196cd968c81e610b3d1aecd06c69467a'
REQUIRED_CARDS = ['archers', 'arrows', 'fireball', 'giant', 'knight', 'mini-pekka', 'minions', 'musketeer']

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

assert DATA.is_file(), f'Missing dataset: {DATA}'
assert CODE_ZIP.is_file(), f'Missing code bundle: {CODE_ZIP}'
assert sha256(DATA) == EXPECTED_DATA_SHA256, 'Dataset SHA-256 mismatch'
assert sha256(CODE_ZIP) == EXPECTED_CODE_SHA256, 'Code bundle SHA-256 mismatch'
print('Dataset and code bundle hashes verified.')


In [ ]:
WORK = Path('/content/clashai_colab_runtime')
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as archive:
    archive.extractall(WORK)
assert (WORK / 'pipeline' / 'train_gen.py').is_file()
assert (WORK / 'pipeline' / 'decks' / 'icebow.yaml').is_file()
sys.path.insert(0, str(WORK))
os.chdir(WORK)

import numpy as np
import torch
with np.load(DATA, allow_pickle=False) as z:
    meta = json.loads(str(z['meta']))
    rows = int(len(z['split']))
    train_rows = int((z['split'] == 0).sum())
    heldout_rows = int((z['split'] == 1).sum())
    v3val_rows = int(z['v3val'].sum())
assert meta.get('feature_version') == 3, f"Expected FV3 dataset, got {meta.get('feature_version')}"
assert rows == 22007 and heldout_rows == 1634, (rows, heldout_rows)
missing = sorted(set(REQUIRED_CARDS) - set(meta['card_vocab']))
assert not missing, f'Recorded Training Camp cards missing from model vocabulary: {missing}'
target_card_support = {c: meta['card_counts'].get(c, {}) for c in REQUIRED_CARDS}
assert all(int(target_card_support[c].get('plays', 0)) > 0 for c in REQUIRED_CARDS), target_card_support
assert torch.cuda.is_available(), 'GPU is not enabled. Select Runtime → Change runtime type → GPU and rerun.'
print(json.dumps({
    'gpu': torch.cuda.get_device_name(0),
    'bf16_supported': bool(torch.cuda.is_bf16_supported()),
    'rows': rows, 'train_rows': train_rows, 'heldout_rows': heldout_rows,
    'v3val_rows': v3val_rows, 'decks': len(meta['decks']), 'cards': len(meta['card_vocab']) - 1,
    'required_card_support': target_card_support,
}, indent=2))


In [ ]:
from datetime import datetime, timezone

EPOCHS = 20
SEED = 0
BATCH_SIZE = 256
GRID = 'lattice'
RUN_ID = 'clashai_training_camp_generalist_fv3_colab_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT_DIR = DRIVE / 'runs' / RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=False)
AMP = 'bf16' if torch.cuda.is_bf16_supported() else 'none'
command = [
    sys.executable, '-m', 'pipeline.train_gen',
    '--data', str(DATA), '--feature-version', '3',
    '--out-dir', str(OUT_DIR), '--tag', RUN_ID,
    '--seed', str(SEED), '--epochs', str(EPOCHS), '--bs', str(BATCH_SIZE),
    '--grid', GRID, '--select', 'cell_tile_top1', '--val-sample', '0', '--amp', AMP,
]
print('Starting fresh Colab training:', ' '.join(command))
process = subprocess.Popen(command, cwd=WORK, env={**os.environ, 'PYTHONPATH': str(WORK)},
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='')
return_code = process.wait()
if return_code:
    raise RuntimeError(f'train_gen exited with status {return_code}; any completed epoch checkpoint remains in {OUT_DIR}')
CHECKPOINT = OUT_DIR / f'gen_{RUN_ID}_s{SEED}.pt'
assert CHECKPOINT.is_file(), f'Trainer completed without a checkpoint: {CHECKPOINT}'
print('Saved checkpoint:', CHECKPOINT)


In [ ]:
EVAL_JSON = OUT_DIR / 'heldout_eval.json'
eval_command = [
    sys.executable, '-m', 'pipeline.eval_gen',
    '--ckpt', str(CHECKPOINT), '--data', str(DATA),
    '--out', str(EVAL_JSON), '--val-sample', '0',
]
subprocess.run(eval_command, cwd=WORK, env={**os.environ, 'PYTHONPATH': str(WORK)}, check=True)
evaluation = json.loads(EVAL_JSON.read_text())
run_manifest = {
    'run_id': RUN_ID, 'status': 'completed', 'seed': SEED, 'epochs': EPOCHS,
    'batch_size': BATCH_SIZE, 'grid': GRID, 'amp': AMP,
    'device': torch.cuda.get_device_name(0),
    'dataset_path': str(DATA), 'dataset_sha256': sha256(DATA),
    'dataset_rows': rows, 'train_rows': train_rows, 'heldout_rows': heldout_rows,
    'feature_version': 3, 'decks': len(meta['decks']), 'card_vocab_size': len(meta['card_vocab']) - 1,
    'target_deck_cards': REQUIRED_CARDS, 'target_card_support': target_card_support,
    'source_corpus_manifest_sha256': SOURCE_CORPUS_MANIFEST_SHA256,
    'source_selection_sha256': SOURCE_SELECTION_SHA256,
    'code_bundle_sha256': sha256(CODE_ZIP),
    'checkpoint_path': str(CHECKPOINT), 'checkpoint_sha256': sha256(CHECKPOINT),
    'evaluation_path': str(EVAL_JSON), 'evaluation': evaluation,
    'live_use_approved': False,
}
manifest_path = OUT_DIR / 'colab_run_manifest.json'
manifest_path.write_text(json.dumps(run_manifest, indent=2))
print(json.dumps({
    'run_id': RUN_ID, 'checkpoint': str(CHECKPOINT),
    'checkpoint_sha256': run_manifest['checkpoint_sha256'],
    'heldout': evaluation['val_all']['model'],
    'manifest': str(manifest_path),
}, indent=2))
